# PaleographIA: continuidade dos dados

Conferência executável do bloco de 1795 após a primeira entrega. Preparado em 03/11/2026.

Esta etapa documenta o conjunto atual e as transformações já registradas. A revisão humana das transcrições e a avaliação científica de ajuste fino continuam pendentes. O notebook lê este checkout e suas fontes locais, usa somente a biblioteca padrão do Python e grava um relatório em `output/continuidade_20261103/`. Não executa reconhecedores ou treinamento nem altera revisões.

Execute da raiz do projeto ou desta pasta. Em outro computador, restaure as fontes conforme o guia de expansão do corpus antes da execução. Consulte o README desta pasta para os requisitos de PI IV e PLN recuperados dos arquivos originais.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import platform

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists() and (p / 'src/paleographia').exists()), None)
if RAIZ is None:
    raise RuntimeError('Abra o notebook dentro do checkout do PaleographIA.')
PACOTE = RAIZ / 'docs/revisao_humana/sessao_1795/sessao_20261103/exportacao_apos_observacao_038'
LINHAS = [json.loads(l) for l in (PACOTE / 'lines.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
REVISAO = json.loads((PACOTE / 'provenance.json').read_text(encoding='utf-8'))['revision']
print('Python:', platform.python_version())
print('Snapshot:', REVISAO)
print('Linhas:', len(LINHAS))
print('Geometria:', dict(Counter(l['geometry_status'] for l in LINHAS)))
print('Texto:', dict(Counter(l['review_status'] for l in LINHAS)))
PAGINAS = [json.loads(l) for l in (PACOTE / 'pages.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
print('Correspondência da página:', PAGINAS[0]['correspondence_status'])
print('Descrição da cobertura:', PAGINAS[0]['transcribed_coverage'][0]['review_status'])
print('Referência completa da região:', PAGINAS[0]['transcribed_coverage'][0]['complete'])


Python: 3.12.14
Snapshot: 20261103T163509_58d6a0ee
Linhas: 8
Geometria: {'validated': 8}
Texto: {'validated': 7, 'pending_review': 1}
Correspondência da página: validated
Descrição da cobertura: validated
Referência completa da região: False


## Integridade das entradas

Cada linha vincula imagem original, caixa, recorte e rascunho diplomático. Os hashes conferem identidade dos arquivos, sem certificar a leitura paleográfica. A primeira linha usa a caixa aprovada de 28/10, em lugar da preparação inicial de 23/10.

In [2]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

INTEGRIDADE = []
for linha in LINHAS:
    for tipo, campo, hash_campo in [('fonte', 'source_image', 'source_sha256'), ('recorte', 'crop_image', 'crop_sha256')]:
        caminho = (RAIZ / linha[campo]).resolve()
        if not caminho.is_relative_to(RAIZ):
            raise ValueError('Caminho fora do checkout: ' + linha[campo])
        existe = caminho.is_file()
        esperado = linha.get(hash_campo)
        obtido = sha256(caminho) if existe else None
        INTEGRIDADE.append({'line_id': linha['line_id'], 'tipo': tipo,
                            'path': linha[campo], 'existe': existe,
                            'hash_declarado': esperado, 'hash_obtido': obtido,
                            'confere': bool(esperado and existe and esperado == obtido)})
print('Vínculos fonte/recorte conferidos:', sum(r['confere'] for r in INTEGRIDADE), '/', len(INTEGRIDADE))
problemas = [r for r in INTEGRIDADE if not r['confere']]
if problemas:
    print(json.dumps(problemas, ensure_ascii=False, indent=2))
    raise ValueError('Restaure ou confira os arquivos antes de continuar.')

Vínculos fonte/recorte conferidos: 16 / 16


## Rascunhos atuais e revisão pendente

Os textos abaixo são rascunhos assistidos, preservados literalmente. A tabela não aprova nenhuma linha. As decisões localizadas sobre Aos, do Juiz, Presidente e a vírgula após Paulo constam no histórico. Restam os pontos do roteiro de conferência textual e a aprovação integral.

In [3]:
for n, linha in enumerate(sorted(LINHAS, key=lambda l: l['reading_order']), 1):
    print(str(n) + '. ' + linha['text_diplomatic'])
    print('   Caixa:', linha['bbox_xyxy'], '| texto:', linha['review_status'], '| geometria:', linha['geometry_status'])
    if linha.get('exclusion_reason'):
        print('   Exclusão:', linha['exclusion_reason'])
APROVADAS_TEXTO_GEOMETRIA = [l for l in LINHAS if l['review_status'] == 'validated'
                           and l['geometry_status'] == 'validated' and not l.get('exclusion_reason')]
print('Linhas que passam os critérios textuais e geométricos mínimos:', len(APROVADAS_TEXTO_GEOMETRIA))
print('Associação e cobertura parcial confirmadas. Treinamento científico ainda exige protocolo e partições documentais.')

1. Aos vinte e cinco dias do mes de junho de mil sete
   Caixa: [95, 305, 1030, 377] | texto: validated | geometria: validated
2. centos e noventa e cinco annos nesta Villa de Nˢᵃ
   Caixa: [95, 374, 1030, 421] | texto: validated | geometria: validated
3. senhora do Desterro de jundiahy comarca da cidade
   Caixa: [95, 416, 1030, 468] | texto: validated | geometria: validated
4. de São Paulo, em casas de morada do Juiz Presidente
   Caixa: [95, 466, 1030, 517] | texto: validated | geometria: validated
5. Salvador de Oliveira Prado, onde se achavão presentes
   Caixa: [95, 514, 1030, 566] | texto: validated | geometria: validated
6. os officiaes da Camara em acto de Vereança, trat-
   Caixa: [95, 563, 1030, 619] | texto: validated | geometria: validated
7. ando sobre o bem commum do povo, na mesma se
   Caixa: [95, 614, 1030, 669] | texto: validated | geometria: validated
8. abriu huma carta do Snr Bernardo José de Lorena
   Caixa: [95, 666, 1030, 721] | texto: pending_review | geometri

## Transformações e decisões já realizadas

| Operação registrada | Evidência | Limite |
| --- | --- | --- |
| Organização das fontes e identificação por hash | Manifests do corpus v0.2 | Ocorrências de imagem não equivalem a documentos independentes |
| Seleção de bloco e recorte manual de oito linhas | Caixas e recortes do snapshot atual | Cobertura parcial da página |
| Ampliação superior do primeiro recorte | Caixa atual `[95,305,1030,377]`, decisão humana 026 | Aprovação do recorte não aprova o texto |
| Separação de edição modernizada e rascunho diplomático | Campos textuais distintos | Edição não pode ser copiada como referência histórica sem conferência |
| Revisões locais com autoria e histórico | Snapshots e observações de 26–28/10 | Sete textos estão validados e a linha 8 permanece provável e pendente |

Este notebook não aplica novas transformações de imagem, transcrição ou estados. A normalização NFC e o colapso de espaços pertencem ao protocolo de comparação do Notebook 1 e devem ser explicitados na próxima avaliação.

Atualização de 03/11: linhas 4 e 6 validadas por confirmação humana; Nˢᵃ e Snr incorporados conforme as descrições do revisor; abriu mantido como provável e explicitamente excluído como alvo aprovado. A edição de apoio e os recortes foram preservados.


Após a observação 037, todas as linhas 1–7 estão validadas. A linha 8 permanece provável e excluída como alvo aprovado, sem alteração da dúvida. Correspondência e cobertura documental continuam em etapa própria.


Após a observação 038, a associação ao fólio 91r da sessão de 25/06/1795 e a descrição da cobertura parcial de oito linhas estão validadas. A referência da região permanece incompleta pela dúvida da oitava linha. Os sete pares revisados estão no pacote dataset_revisado_1795, sem atribuição científica de partições.


In [4]:
PROTOCOLO = json.loads((RAIZ / 'data/manifests/corpus_expandido_v02/partitions.json').read_text(encoding='utf-8'))
print('Situação do protocolo:', PROTOCOLO['status'])
for grupo in PROTOCOLO['groups']:
    print(grupo['candidate_group'], '| independência:', grupo['independence_status'],
          '| uso atual:', grupo['partition'], '| partição científica:', grupo['scientific_split'])
print('Traslado e posturas de 1831 já são desenvolvimento.')
print('Os grupos conhecidos nesta curadoria precisam ser tratados como conhecidos no protocolo de avaliação.')

Situação do protocolo: blocked_pending_independent_reviewed_groups
carta_jose_gomes_1767 | independência: pending_review | uso atual: curation | partição científica: None
contrato_livro12_1763 | independência: pending_review | uso atual: curation | partição científica: None
contrato_livro13_1763 | independência: pending_review | uso atual: curation | partição científica: None
pagamento_thomas_pardo | independência: pending_review | uso atual: curation | partição científica: None
posturas_1831 | independência: pending_review | uso atual: development | partição científica: None
sessao_1795 | independência: pending_review | uso atual: curation | partição científica: None
traslado_criacao_vila | independência: pending_review | uso atual: development | partição científica: None
Traslado e posturas de 1831 já são desenvolvimento.
Os grupos conhecidos nesta curadoria precisam ser tratados como conhecidos no protocolo de avaliação.


## Prontidão para a próxima avaliação

Concluir a referência textual, conferir identificação/cobertura e definir partições por documentos independentes antes de selecionar modelos. A sessão de 1795 isolada não preenche treino, validação e teste científicos. O ensaio técnico histórico de dois passos comprova execução de otimização, sem demonstrar ganho em documentos reservados.

In [5]:
PENDENCIAS = [
    'Manter a linha 8 com abriu provável, alternativa abrio e exclusão explícita como alvo aprovado.',
    'Confirmar independência dos grupos e definir partições científicas com pares aptos.',
]
RELATORIO = {
    'data_conferencia': '2026-11-03', 'tipo': 'prontidao_dataset_sem_treino',
    'snapshot': REVISAO, 'linhas': len(LINHAS), 'textos_pendentes': sum(l['review_status'] == 'pending_review' for l in LINHAS),
    'geometrias_validadas': sum(l['geometry_status'] == 'validated' for l in LINHAS),
    'textos_validados': sum(l['review_status'] == 'validated' for l in LINHAS),
    'pares_revisados_validos_por_linha': len(APROVADAS_TEXTO_GEOMETRIA), 'particoes_cientificas_prontas': 0,
    'experimento_cientifico_pronto': False,
    'novas_inferencias': 0, 'novo_treinamento': False,
    'integridade': INTEGRIDADE, 'pendencias': PENDENCIAS,
}
DESTINO = RAIZ / 'output/continuidade_20261103/prontidao_dataset.json'
DESTINO.parent.mkdir(parents=True, exist_ok=True)
DESTINO.write_text(json.dumps(RELATORIO, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Relatório:', DESTINO.relative_to(RAIZ))
for p in PENDENCIAS:
    print('-', p)

Relatório: output\continuidade_20261103\prontidao_dataset.json
- Manter a linha 8 com abriu provável, alternativa abrio e exclusão explícita como alvo aprovado.
- Confirmar independência dos grupos e definir partições científicas com pares aptos.


## Comentários

Ferramenta: Codex, da OpenAI.

Utilização: recuperação dos requisitos enviados, organização deste notebook, conferência dos vínculos e hashes e execução sequencial das células. As leituras humanas pertencem ao revisor identificado nos registros. A execução não promove estados de revisão nem demonstra desempenho de reconhecimento.

Fontes: snapshot da sessão de 1795 de 28/10, manifests do corpus v0.2, roteiro textual atual e documentos acadêmicos identificados em `fontes_requisitos.json`.